# Mamba-Spectral: Spectral Analysis of Mamba SSM

This notebook demonstrates the core functionality of the `mamba-spectral` library for analyzing Mamba language models through spectral theory.

## Key Concepts

1. **Spectral Radius (ρ)**: Determines memory capacity. ρ ≈ 1 means long-term memory preservation.
2. **Reachability Gramian (W_R)**: Determines if states are reachable from input.
3. **Spectral Horizon**: The maximum reasoning depth before information is lost.
4. **SpectralGuard**: Defense against HiSPA attacks that cause spectral collapse.

## Setup

In [ ]:
# Install if needed
# !pip install torch numpy scipy matplotlib seaborn scikit-learn tqdm transformers
# !pip install mamba-ssm  # Optional: for GPU-accelerated Mamba

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
%matplotlib inline

# Import mamba-spectral
from mamba_spectral import (
    MambaWrapper,
    StateExtractor,
    SpectralAnalyzer,
    ReachabilityGramian,
    HorizonPredictor,
    SpectralGuard,
)
from mamba_spectral.visualization import (
    plot_eigenvalue_spectrum,
    plot_spectral_radius_trajectory,
    plot_gramian_heatmap,
    create_spectral_dashboard,
)

print("Mamba-Spectral loaded successfully!")

## 1. Basic Eigenvalue Analysis (Without Model)

First, let's understand the spectral analysis with synthetic data.

In [ ]:
# Create a synthetic diagonal A matrix (like Mamba)
d_inner = 64   # Inner dimension
d_state = 16   # SSM state dimension

# In Mamba: A = -exp(A_log), ensuring stability
A_log = torch.randn(d_inner, d_state) - 1.0  # Shift for stability
A = -torch.exp(A_log)

print(f"A matrix shape: {A.shape}")
print(f"A range: [{A.min():.4f}, {A.max():.4f}]")

In [ ]:
# Discretize using Zero-Order Hold
delta = 0.01  # Discretization step
A_bar = torch.exp(delta * A)

# For diagonal A, eigenvalues ARE the diagonal elements
eigenvalues = A_bar.flatten().numpy()

# Compute spectral radius
spectral_radius = np.max(np.abs(eigenvalues))

print(f"Number of eigenvalues: {len(eigenvalues)}")
print(f"Spectral radius ρ(Ā): {spectral_radius:.6f}")
print(f"Mean |λ|: {np.mean(np.abs(eigenvalues)):.6f}")

In [ ]:
# Visualize eigenvalue spectrum
fig = plot_eigenvalue_spectrum(
    eigenvalues,
    title=f"Mamba Eigenvalue Spectrum (ρ = {spectral_radius:.4f})",
    figsize=(8, 8),
)
plt.show()

## 2. Effect of Discretization Step (Δ)

The discretization step Δ controls the effective spectral radius. Larger Δ → eigenvalues closer to unit circle.

In [ ]:
# Vary delta and observe spectral radius
delta_values = np.logspace(-3, 0, 50)  # From 0.001 to 1.0
radii = []

for delta in delta_values:
    A_bar = torch.exp(delta * A)
    eigenvalues = A_bar.flatten().numpy()
    radii.append(np.max(np.abs(eigenvalues)))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(delta_values, radii, 'b-', linewidth=2)
ax.axhline(y=1.0, color='red', linestyle='--', label='Stability boundary')
ax.set_xlabel('Discretization Step Δ', fontsize=12)
ax.set_ylabel('Spectral Radius ρ(Ā)', fontsize=12)
ax.set_title('Effect of Δ on Spectral Radius', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 3. Reachability Gramian Computation

The Gramian tells us which states are reachable (controllable) from input.

In [ ]:
# Create test matrices
n = 16
A_test = torch.eye(n) * 0.9  # Stable diagonal
B_test = torch.randn(n, 1)

# Compute gramian
gramian_calc = ReachabilityGramian(device='cpu')
result = gramian_calc.compute(
    A_test, B_test,
    horizon=50,
    track_singular_values=True,
)

print(f"Gramian shape: {result.gramian.shape}")
print(f"Rank: {result.rank}/{n}")
print(f"Is full rank: {result.is_full_rank}")
print(f"Min singular value: {result.min_singular_value:.2e}")

In [ ]:
# Visualize gramian
fig = plot_gramian_heatmap(
    result.gramian,
    title="Reachability Gramian W_R",
    log_scale=True,
)
plt.show()

In [ ]:
# Plot singular value evolution
from mamba_spectral.visualization.spectral_plots import plot_singular_value_trajectory

fig = plot_singular_value_trajectory(
    result.singular_values,
    horizon=50,
    title="Gramian Singular Values Over Horizon",
)
plt.show()

## 4. Simulating Spectral Trajectory During Inference

Simulate how spectral radius changes as we process tokens.

In [ ]:
# Simulate spectral trajectory
# In real model, delta varies per token
seq_len = 100
base_delta = 0.01

# Simulate input-dependent delta (increases with position)
trajectory = []
for t in range(seq_len):
    # Delta varies with position (simplified model)
    delta_t = base_delta * (1 + 0.005 * t + 0.01 * np.random.randn())
    delta_t = max(0.001, delta_t)  # Clamp
    
    A_bar_t = torch.exp(delta_t * A)
    rho_t = torch.max(torch.abs(A_bar_t)).item()
    trajectory.append(rho_t)

# Plot
fig = plot_spectral_radius_trajectory(
    trajectory,
    title="Simulated Spectral Radius During Inference",
    threshold=0.3,
)
plt.show()

## 5. Eigenvalue Clustering (Spectral Engrams)

Eigenvalues may cluster into groups that encode different information types.

In [ ]:
from sklearn.cluster import KMeans
from mamba_spectral.visualization.spectral_plots import plot_eigenvalue_clusters

# Add some structure to eigenvalues for clustering
n_eigvals = 200
# Create 3 clusters
cluster1 = 0.9 * np.exp(1j * np.random.uniform(-0.1, 0.1, 60))
cluster2 = 0.5 * np.exp(1j * np.random.uniform(-0.2, 0.2, 80))
cluster3 = 0.2 * np.exp(1j * np.random.uniform(0.5, 1.5, 60))

eigenvalues_clustered = np.concatenate([cluster1, cluster2, cluster3])

# Cluster
X = np.column_stack([np.real(eigenvalues_clustered), np.imag(eigenvalues_clustered)])
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_[:, 0] + 1j * kmeans.cluster_centers_[:, 1]

# Plot
fig = plot_eigenvalue_clusters(
    eigenvalues_clustered,
    labels,
    centers,
    title="Eigenvalue Clustering (Spectral Engrams)",
)
plt.show()

## 6. SpectralGuard: Defense Simulation

Demonstrate how SpectralGuard detects HiSPA-style attacks.

In [ ]:
# Simulate benign vs adversarial trajectories
np.random.seed(42)

# Benign: stable spectral radius
benign_trajectory = 0.85 + 0.02 * np.random.randn(50)

# Adversarial: sudden drop (spectral collapse)
adversarial_trajectory = np.concatenate([
    0.85 + 0.02 * np.random.randn(20),  # Normal start
    np.linspace(0.85, 0.1, 10),          # Collapse
    0.1 + 0.01 * np.random.randn(20),   # Stay low
])

# Visualize
from mamba_spectral.visualization.trajectory_viz import plot_attack_comparison

fig = plot_attack_comparison(
    benign_trajectory.tolist(),
    adversarial_trajectory.tolist(),
    attack_name="HiSPA",
    title="SpectralGuard: Detecting Hidden State Poisoning",
)
plt.show()

In [ ]:
# Simulate detection logic
def simple_detect_collapse(trajectory, window_size=5, collapse_threshold=0.3):
    """Simple collapse detection."""
    for i in range(len(trajectory) - window_size):
        window = trajectory[i:i + window_size]
        drop = window[0] - window[-1]
        if drop > collapse_threshold:
            return True, i, drop
    return False, None, 0

# Test on both
is_attack_benign, loc_benign, drop_benign = simple_detect_collapse(benign_trajectory)
is_attack_adv, loc_adv, drop_adv = simple_detect_collapse(adversarial_trajectory)

print("Benign prompt:")
print(f"  Attack detected: {is_attack_benign}")

print("\nAdversarial prompt:")
print(f"  Attack detected: {is_attack_adv}")
if is_attack_adv:
    print(f"  Location: token {loc_adv}")
    print(f"  Drop: {drop_adv:.3f}")

## 7. Full Dashboard

In [ ]:
# Create comprehensive dashboard
fig = create_spectral_dashboard(
    eigenvalues,
    trajectory,
    result.gramian,
    title="Mamba Spectral Analysis Dashboard",
)
plt.show()

## 8. Validation Test

In [ ]:
# Run validation to check installation
from mamba_spectral.utils.validation import validation_test

validation_test(verbose=True, run_gpu_tests=False)

## 9. Next Steps

To use with actual Mamba models:

```python
# Load pretrained model
wrapper = MambaWrapper.load_pretrained("state-spaces/mamba-130m")

# Create analyzer
analyzer = SpectralAnalyzer(wrapper)

# Extract and analyze A matrix
A = analyzer.extract_A_matrix(layer_idx=0)

# Track spectral evolution
trajectory = analyzer.track_evolution("Your prompt here...")
```